In [2]:
import base64
import requests
import time
import pandas as pd
import os

# Claude API Key
api_key = ""

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs _Bad Prompt/VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []


for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Getting the base64 string
        base64_image = encode_image(image_path)
        
        headers = {
            "Content-Type": "application/json",
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01"
        }
        
        payload = {
            "model": "claude-3-7-sonnet-20250219",
            "max_tokens": 5000,
            "temperature": 0.0,
            "system": "You are an assistant, skilled in reading and interpreting visually represented data.",
            "messages": [
                {
                    "role": "user",
                    "content": 'I am about to show you an image and ask you a multiple choice question about that image. Select the BEST answer, based only on the chart and not external knowledge.' +
                    'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n'
                 
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": question_text + " " + question_options
                        },
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": base64_image
                            }
                        }
                    ]
                }
            ]
        }
        
        # Add retry logic for overloaded errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                time_start = time.perf_counter()
                response_raw = requests.post(
                    "https://api.anthropic.com/v1/messages",
                    headers=headers,
                    json=payload
                )
                time_end = time.perf_counter()
                
                response = response_raw.json()
                
                # If we get an overloaded error, wait and retry
                if 'error' in response and response['error'].get('type') == 'overloaded_error':
                    if retry < max_retries - 1:  # Don't sleep on the last retry
                        print(f"\nAPI overloaded, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                        time.sleep(retry_delay)
                        continue
                
                break  # If we get here, we either got a good response or a different error
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        print("\nAPI Response:", response)  # Debug print
        
        if 'error' in response:
            gpt_answer = f"{response['error']['type']}: {response['error']['message']}"
            is_correct = "N/A"
        else:
            # Extract the answer after "Correct Answer: "
            full_response = response["content"][0]["text"].strip()
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs _Bad Prompt/Results/1_Claude 3.7_Final Results with Explanation/"
os.makedirs(results_dir, exist_ok=True)

results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")


Processing question 1:
{'id': 1, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_1', 'question: ': 'What was the price of a barrel of oil in February 2015? ', 'option:': '$39.72; $57.36; $48.90; $62.85', 'correct': '$48.90 '}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: What was the price of a barrel of oil in February 2015? 
Options: $39.72; $57.36; $48.90; $62.85
Correct answer: $48.90

API Response: {'id': 'msg_015eGvyiRuXv6CN7ejsVp15b', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the chart "Monthly Oil Price History in 2015," I can see the price points for each month plotted on the line graph. For February 2015, the data point appears to be at approximately $48.90 per barrel. This is the second point on the graph, following January.\n\nCorrect Answer: $48.90'}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 4


Processing question 8:
{'id': 8, 'dropped': 'no', 'vis': 'VLAT_b', 'item': 'b_3', 'question: ': 'What is the range of the average internet speed in Asia?', 'option:': '0 - 25 Mbps; 9 - 18 Mbps; 3 - 20 Mbps; 4 - 7 Mbps', 'correct': '3 - 20 Mbps'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_b.png
Question: What is the range of the average internet speed in Asia?
Options: 0 - 25 Mbps; 9 - 18 Mbps; 3 - 20 Mbps; 4 - 7 Mbps
Correct answer: 3 - 20 Mbps

API Response: {'id': 'msg_01BWHY2dB9TjjKMq5bK5bMU7', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the bar chart titled "Average Internet Speeds in Asia," I can see the internet speeds for various Asian countries measured in Mbps. The lowest value appears to be Hong Kong at around 3 Mbps, and the highest value is South Korea at about 20 Mbps. Therefore, the range of average internet speeds shown in the chart is ap


Processing question 15:
{'id': 15, 'dropped': 'no', 'vis': 'VLAT_c', 'item': 'c_6', 'question: ': 'The ratio of the cost of Soda to the cost of Water in Orlando is higher than that of Washington D.C.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_c.png
Question: The ratio of the cost of Soda to the cost of Water in Orlando is higher than that of Washington D.C.
Options: True; False
Correct answer: TRUE

API Response: {'id': 'msg_013SnEXKjS9vM9FqDMAC24o2', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the chart, I need to find the ratio of Soda to Water costs in both Orlando and Washington D.C.\n\nFor Orlando:\n- Soda cost (orange section): approximately $15\n- Water cost (blue section): approximately $10\n- Ratio of Soda to Water in Orlando: 15/10 = 1.5\n\nFor Washington D.C.:\n- Soda cost (orange section): appro


API Response: {'id': 'msg_01WC4XfDBEfBaX1v4BGZ7Jf6', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "Based on the pie chart, Apple has a global smartphone market share of 28.0%, while Huawei has a market share of 15.0%. Since Apple's market share (28.0%) is larger than Huawei's market share (15.0%), the statement is true.\n\nCorrect Answer: True"}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 432, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 73}}

Answer: True
Time taken: 2.14 seconds
Correct? True

Processing question 22:
{'id': 22, 'dropped': 'no', 'vis': 'VLAT_f', 'item': 'f_1', 'question: ': 'How many people have rated the taxi between 4.0 and 4.2?', 'option:': '145; 183; 250; 310', 'correct': '310'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_f.png
Question: How many people have rated the taxi bet


Processing question 28:
{'id': 28, 'dropped': 'no', 'vis': 'VLAT_g', 'item': 'g_2', 'question: ': 'What is the height for the tallest person among the 85 males?', 'option:': '145.3 cm; 175.3 cm; 182 cm; 197.1 cm', 'correct': '197.1 cm'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_g.png
Question: What is the height for the tallest person among the 85 males?
Options: 145.3 cm; 175.3 cm; 182 cm; 197.1 cm
Correct answer: 197.1 cm

API Response: {'id': 'msg_01NZYscGS9EgSRq2Fwku1QVr', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the scatter plot of Height vs. Weight for 85 males, I need to find the tallest person (the rightmost point on the x-axis).\n\nBased on the x-axis (Height in cm), the rightmost data point appears to be at approximately 197.1 cm.\n\nCorrect Answer: 197.1 cm'}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 40


API Response: {'id': 'msg_017zMqS8KvBZCdu1ATA4E262', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the scatter plot showing Height vs. Weight of 85 Males, I can see that in the height range of 185-190 cm, there are several data points with different weight values. The weights appear to range from approximately 55-60 kg to about 100 kg in this height range. The data points are clearly at different vertical positions, indicating different weights.\n\nCorrect Answer: False'}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 386, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 89}}

Answer: False
Time taken: 2.94 seconds
Correct? True

Processing question 35:
{'id': 35, 'dropped': 'no', 'vis': 'VLAT_i', 'item': 'i_1', 'question: ': 'What was the average price of a pound of coffee beans in September 2013?', 'option:': '$4.90; $5.00; $5.10; $5.50', '


Processing question 41:
{'id': 41, 'dropped': 'no', 'vis': 'VLAT_j', 'item': 'j_2', 'question: ': 'About what was the ratio of the number of girls named ‘Olivia’ to those named ‘Isla’ in 2014 in the UK?', 'option:': '1 to 1; 1 to 7; 1 to 15; 1 to 20', 'correct': '1 to 1'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_j.png
Question: About what was the ratio of the number of girls named ‘Olivia’ to those named ‘Isla’ in 2014 in the UK?
Options: 1 to 1; 1 to 7; 1 to 15; 1 to 20
Correct answer: 1 to 1

API Response: {'id': 'msg_01HtkoEE8iUzmUHhrd2yHJ37', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the chart for 2014 (the rightmost point on the x-axis), I need to determine the ratio between Olivia and Isla.\n\nFrom the stacked area chart:\n- Olivia is represented by the lightest blue area at the top\n- Isla is represented by the medium blue area in the middle\


Processing question 47:
{'id': 47, 'dropped': 'no', 'vis': 'VLAT_h', 'item': 'h_1', 'question: ': 'What is the total length of the metro system in C?', 'option:': '330 km; 400 km; 530 km; 560 km', 'correct': '330 km'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_h.png
Question: What is the total length of the metro system in C?
Options: 330 km; 400 km; 530 km; 560 km
Correct answer: 330 km

API Response: {'id': 'msg_013EoK2GDPEuSGPoQtypQyxr', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the chart, I can see City C is positioned at approximately 330 km on the y-axis (Total System Length). The bubble for City C appears to be around the 330 km mark.\n\nCorrect Answer: 330 km'}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 784, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 56}}

Answer: 330 km



Processing question 53:
{'id': 53, 'dropped': 'no', 'vis': 'VLAT_h', 'item': 'h_7', 'question: ': 'In general, the ridership of the metro system increases as the number of stations increases.', 'option:': 'True; False', 'correct': 'FALSE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_h.png
Question: In general, the ridership of the metro system increases as the number of stations increases.
Options: True; False
Correct answer: FALSE

API Response: {'id': 'msg_01Szt54fLMf7hZKeTWdWJUZz', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': "Looking at the chart, I need to determine if ridership (represented by bubble size) increases as the number of stations (x-axis) increases.\n\nExamining the data points, I can see that while some cities with more stations do have higher ridership (like City B at around 420 stations with the largest bubble), there isn't a consistent pattern. 


Processing question 59:
{'id': 59, 'dropped': 'no', 'vis': 'VLAT_l', 'item': 'l_2', 'question: ': 'For which website was the number of unique visitors the largest in 2010?', 'option:': 'Facebook; Amazon; Bing; Google', 'correct': 'Google'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_l.png
Question: For which website was the number of unique visitors the largest in 2010?
Options: Facebook; Amazon; Bing; Google
Correct answer: Google

API Response: {'id': 'msg_01PzU8YTc1WgBmCbH3fijY4b', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-7-sonnet-20250219', 'content': [{'type': 'text', 'text': 'Looking at the image, I can see a treemap visualization showing various websites categorized by type (Retail, Financial, Social Media, Search, News, Computer). The size of each rectangle appears to represent the number of unique visitors.\n\nBased on the relative sizes of the rectangles in the chart, Google (in the Search category) appears t